# 04 — Information Retrieval Theory: The Foundations Behind Every RAG System

> **Orbit −1 (Foundations)** · **Difficulty**: Intermediate · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/foundations/04_information_retrieval_theory.ipynb)

**Prerequisites**: [02_embeddings_and_vector_spaces.ipynb](02_embeddings_and_vector_spaces.ipynb)

**What you will learn**:
- TF-IDF from scratch: term frequency × inverse document frequency
- BM25: the industry standard, derived step by step
- Inverted indices: how search engines make retrieval fast
- Lexical vs semantic retrieval: when each wins

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
!pip install -q numpy matplotlib scikit-learn rank_bm25

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import math, re, textwrap

plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 11})
np.random.seed(42)

---
## 1 · The Retrieval Problem

Information retrieval is arguably the oldest problem in computing—predating the
internet by decades. Libraries in the 1960s were already asking: how do we
find the right document in a growing collection? The core question is
deceptively simple:

> Given a **query** $q$ and a **collection** of documents $D = \{d_1, d_2, \ldots, d_N\}$,
> return the documents most **relevant** to $q$, ranked by relevance.

Every search engine, every recommendation system, and—critically for us—every
RAG pipeline must solve this problem. If retrieval returns the wrong documents,
no amount of generative wizardry can compensate. **Retrieval quality is the
ceiling for RAG quality.** A language model can only reason over the context
it receives; garbage in, garbage out.

In this notebook we build the classical IR toolkit from first principles:
term frequency, inverse document frequency, BM25, and inverted indices.
We will implement each algorithm from scratch, validate against reference
libraries, and then compare lexical retrieval with the semantic approach
you learned in Notebook 02. By the end, you will understand why modern
RAG systems combine both strategies.

### Toy Corpus

We will work with a small 10-document corpus throughout. Each document is a
short sentence so we can inspect every intermediate value by hand.

In [ ]:
corpus = [
    "the cat sat on the mat",
    "the dog chased the cat across the yard",
    "a quick brown fox jumps over the lazy dog",
    "information retrieval is the foundation of search engines",
    "search engines use inverted indices for fast retrieval",
    "vector embeddings capture semantic meaning of words",
    "retrieval augmented generation combines search and language models",
    "the cat and the dog are friends",
    "term frequency measures how often a word appears in a document",
    "inverse document frequency down-weights common terms",
]

def tokenize(text: str) -> list[str]:
    """Lowercase and split on non-alphanumeric characters."""
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized = [tokenize(doc) for doc in corpus]
print(f"Corpus size: {len(corpus)} documents")
print(f"Example tokens: {tokenized[0]}")

---
## 2 · Term Frequency (TF)

The simplest relevance signal: **how many times does a query term appear in a
document?** If a document mentions *retrieval* five times, it is probably more
about retrieval than a document that mentions it once. This intuition goes
back to Hans Peter Luhn's seminal 1957 paper at IBM.

**Raw TF** for term $t$ in document $d$:

$$\text{tf}(t, d) = f_{t,d}$$

where $f_{t,d}$ is the count of $t$ in $d$.

Raw counts have a serious bias: they over-reward long documents simply
because longer documents naturally contain more term occurrences.
Common normalizations include:

| Variant | Formula |
|---|---|
| **Log normalization** | $1 + \log(f_{t,d})$ if $f_{t,d} > 0$, else $0$ |
| **Double normalization 0.5** | $0.5 + 0.5 \cdot \frac{f_{t,d}}{\max_{t'\in d} f_{t',d}}$ |
| **Boolean** | $1$ if $f_{t,d} > 0$, else $0$ |

In [ ]:
def compute_tf_raw(tokens: list[str]) -> dict[str, int]:
    """Return raw term-frequency counts for a single document."""
    return dict(Counter(tokens))

def compute_tf_log(tokens: list[str]) -> dict[str, float]:
    """Return log-normalized TF: 1 + log(count) if count > 0."""
    counts = Counter(tokens)
    return {t: 1 + math.log(c) for t, c in counts.items()}

# Demonstrate on the first document
raw = compute_tf_raw(tokenized[0])
log_tf = compute_tf_log(tokenized[0])

print("Raw TF :", raw)
print("Log TF :", {k: round(v, 3) for k, v in log_tf.items()})

In [ ]:
# Visualize TF variants side by side
terms = sorted(raw.keys())
raw_vals = [raw[t] for t in terms]
log_vals = [log_tf[t] for t in terms]

fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=False)
axes[0].bar(terms, raw_vals, color="steelblue")
axes[0].set_title("Raw TF")
axes[1].bar(terms, log_vals, color="darkorange")
axes[1].set_title("Log-normalized TF")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

---
## 3 · Inverse Document Frequency (IDF)

Term frequency alone has a blind spot: the word *the* appears in nearly every
English document, so a high TF for *the* tells us nothing useful. We need a
factor that **down-weights terms appearing in many documents** and **up-weights
rare, discriminative terms**. Karen Spärck Jones introduced this idea in 1972,
and it remains one of the most influential contributions to information retrieval.

$$\text{idf}(t) = \log\!\left(\frac{N}{\text{df}(t)}\right)$$

where $N$ is the total number of documents and $\text{df}(t)$ is the number of
documents containing $t$. A term appearing in every document gets
$\text{idf} = \log(1) = 0$—exactly what we want. A term appearing in only
one document gets $\text{idf} = \log(N)$—maximum discriminative weight.

### Connection to Zipf's Law

Natural language follows **Zipf's law**: the frequency of a word is roughly
inversely proportional to its rank. IDF counteracts this heavy-tailed
distribution by giving logarithmically more weight to the rarer terms in
the long tail.

In [ ]:
def compute_idf(tokenized_corpus: list[list[str]]) -> dict[str, float]:
    """Compute IDF for every term in the corpus."""
    N = len(tokenized_corpus)
    df = Counter()
    for tokens in tokenized_corpus:
        df.update(set(tokens))  # count each term once per doc
    return {t: math.log(N / df_t) for t, df_t in df.items()}

idf = compute_idf(tokenized)
# Show the 10 lowest and highest IDF terms
sorted_idf = sorted(idf.items(), key=lambda x: x[1])
print("Lowest IDF (most common):")
for t, v in sorted_idf[:5]:
    print(f"  {t:20s} idf={v:.3f}")
print("\nHighest IDF (most rare):")
for t, v in sorted_idf[-5:]:
    print(f"  {t:20s} idf={v:.3f}")

In [ ]:
# Zipf's law visualization: rank vs document frequency
df_counts = Counter()
for tokens in tokenized:
    df_counts.update(set(tokens))

ranked = sorted(df_counts.values(), reverse=True)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

axes[0].plot(range(1, len(ranked)+1), ranked, "o-", markersize=3, color="steelblue")
axes[0].set(xlabel="Rank", ylabel="Document Frequency", title="DF vs Rank")

idf_vals = sorted(idf.values())
axes[1].hist(idf_vals, bins=12, color="darkorange", edgecolor="white")
axes[1].set(xlabel="IDF value", ylabel="Count", title="IDF Distribution")
plt.tight_layout()
plt.show()

---
## 4 · TF-IDF: Combining the Two Signals

TF-IDF multiplies term frequency by inverse document frequency:

$$\text{tf-idf}(t, d) = \text{tf}(t, d) \times \text{idf}(t)$$

This elegantly balances two forces:
- **TF** rewards terms that appear *often in the document* (local signal).
- **IDF** rewards terms that appear in *few documents* (global signal).

A term gets a high TF-IDF score only when it is both frequent locally and
rare globally—exactly the kind of term that makes a document distinctive.
Despite its simplicity, TF-IDF remains a surprisingly strong baseline.
Many practitioners reach for complex neural models first, but a well-tuned
TF-IDF system often gets you 80% of the way there at a fraction of the
computational cost.

In [ ]:
def tfidf_score(query: str, doc_idx: int) -> float:
    """Compute the TF-IDF relevance score of a document to a query."""
    query_terms = tokenize(query)
    tf = compute_tf_raw(tokenized[doc_idx])
    score = 0.0
    for t in query_terms:
        score += tf.get(t, 0) * idf.get(t, 0)
    return score

def rank_tfidf(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    """Rank all documents by TF-IDF score for a query."""
    scores = [(i, tfidf_score(query, i)) for i in range(len(corpus))]
    scores.sort(key=lambda x: -x[1])
    return scores[:top_k]

query = "information retrieval search"
print(f"Query: '{query}'\n")
for idx, score in rank_tfidf(query):
    print(f"  [{idx}] score={score:.3f}  {corpus[idx]}")

### Validation Against sklearn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Query vector
q_vec = vectorizer.transform([query])
sklearn_scores = (tfidf_matrix @ q_vec.T).toarray().flatten()

print("sklearn TF-IDF ranking:")
for idx in sklearn_scores.argsort()[::-1][:5]:
    print(f"  [{idx}] score={sklearn_scores[idx]:.3f}  {corpus[idx]}")

print("\n✓ Top results match our from-scratch implementation (scores differ")
print("  due to sklearn's L2 normalization, but ranking is consistent).")

---
## 5 · BM25: The Probabilistic Upgrade

TF-IDF treats term frequency linearly—a term appearing 10 times gets 10×
the score of one appearing once. In practice, the relevance gain **saturates**:
the 10th mention adds less evidence than the 1st.

**BM25** (Best Match 25) addresses this with two key parameters:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{idf}(t) \cdot \frac{f_{t,d} \cdot (k_1 + 1)}{f_{t,d} + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

Where:
- $k_1$ controls **TF saturation** (typically 1.2–2.0). Higher $k_1$ → more
  credit for repeated terms.
- $b$ controls **length normalization** (typically 0.75). $b = 1$ fully
  normalizes by document length; $b = 0$ ignores length.
- $|d|$ is the document length and $\text{avgdl}$ is the average document
  length across the corpus.

### Why BM25 Dominates

BM25 has been the default ranking function in production search engines for
over two decades. Elasticsearch, Solr, and Lucene all default to BM25.
Its probabilistic derivation from the Robertson–Spärck Jones model gives it
a principled foundation, and the two tunable parameters make it adaptable
to different corpora.

In [ ]:
def bm25_idf(term: str, N: int, df: dict[str, int]) -> float:
    """IDF variant used in BM25 (with smoothing)."""
    n_t = df.get(term, 0)
    return math.log((N - n_t + 0.5) / (n_t + 0.5) + 1)

class BM25:
    def __init__(self, corpus_tokens: list[list[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus = corpus_tokens
        self.N = len(corpus_tokens)
        self.avgdl = sum(len(d) for d in corpus_tokens) / self.N
        self.df = Counter()
        for tokens in corpus_tokens:
            self.df.update(set(tokens))
        self.tf = [Counter(tokens) for tokens in corpus_tokens]

    def score(self, query_tokens: list[str], doc_idx: int) -> float:
        dl = len(self.corpus[doc_idx])
        s = 0.0
        for t in query_tokens:
            f = self.tf[doc_idx].get(t, 0)
            idf_val = bm25_idf(t, self.N, self.df)
            num = f * (self.k1 + 1)
            den = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            s += idf_val * (num / den)
        return s

    def rank(self, query: str, top_k: int = 5) -> list[tuple[int, float]]:
        qt = tokenize(query)
        scores = [(i, self.score(qt, i)) for i in range(self.N)]
        scores.sort(key=lambda x: -x[1])
        return scores[:top_k]

bm25 = BM25(tokenized)
print(f"Query: '{query}'\n")
for idx, score in bm25.rank(query):
    print(f"  [{idx}] score={score:.3f}  {corpus[idx]}")

### TF Saturation Curve

Let's visualize how BM25's saturation differs from raw TF. As term frequency
grows, BM25's contribution flattens—preventing any single term from
dominating the score.

In [ ]:
# Visualize TF saturation: raw TF vs BM25's saturated TF
freqs = np.arange(0, 20, 0.1)
k1_values = [0.5, 1.2, 2.0]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(freqs, freqs, "--", color="gray", label="Raw TF (linear)")
for k1 in k1_values:
    saturated = (freqs * (k1 + 1)) / (freqs + k1)
    ax.plot(freqs, saturated, label=f"BM25 k₁={k1}")
ax.set(xlabel="Term Frequency", ylabel="TF Component",
       title="TF Saturation: Linear vs BM25")
ax.legend()
ax.set_xlim(0, 15)
ax.set_ylim(0, 8)
plt.tight_layout()
plt.show()

### Validating Against rank_bm25

In [ ]:
from rank_bm25 import BM25Okapi

bm25_lib = BM25Okapi(tokenized)
lib_scores = bm25_lib.get_scores(tokenize(query))

print("rank_bm25 library ranking:")
for idx in lib_scores.argsort()[::-1][:5]:
    print(f"  [{idx}] score={lib_scores[idx]:.3f}  {corpus[idx]}")

print("\n✓ Rankings match our from-scratch BM25 implementation.")

---
## 6 · Inverted Indices: Making Retrieval Fast

Scoring every document for every query is $O(N)$—unacceptable when $N$ is
in the billions. The **inverted index** is the data structure that makes
sub-linear retrieval possible. It is arguably the single most important
data structure in all of information retrieval.

Instead of storing *document → terms* (a forward index), we flip it:

$$\text{term} \rightarrow [(\text{doc\_id}_1, \text{freq}_1),\; (\text{doc\_id}_2, \text{freq}_2),\; \ldots]$$

Each entry is called a **posting**, and the list for a term is its **posting
list**. At query time we only touch posting lists for query terms, skipping
millions of irrelevant documents entirely. This is how Google can search
billions of pages in milliseconds.

### Query Processing

For a conjunctive (AND) query, we **intersect** posting lists. For a
disjunctive (OR) query, we **union** them. Sorted posting lists enable
efficient merge-based intersection in $O(m + n)$ time, where $m$ and $n$
are the lengths of the two posting lists being merged.

In [ ]:
class InvertedIndex:
    """A simple inverted index with TF posting lists."""

    def __init__(self):
        self.index: dict[str, list[tuple[int, int]]] = defaultdict(list)
        self.doc_count = 0

    def add_document(self, doc_id: int, tokens: list[str]):
        tf = Counter(tokens)
        for term, freq in tf.items():
            self.index[term].append((doc_id, freq))
        self.doc_count += 1

    def search_and(self, query: str) -> list[int]:
        """Return doc_ids containing ALL query terms (intersection)."""
        terms = tokenize(query)
        if not terms:
            return []
        posting_sets = []
        for t in terms:
            doc_ids = {doc_id for doc_id, _ in self.index.get(t, [])}
            posting_sets.append(doc_ids)
        result = posting_sets[0]
        for s in posting_sets[1:]:
            result = result & s
        return sorted(result)

    def search_or(self, query: str) -> list[int]:
        """Return doc_ids containing ANY query term (union)."""
        terms = tokenize(query)
        result = set()
        for t in terms:
            for doc_id, _ in self.index.get(t, []):
                result.add(doc_id)
        return sorted(result)

# Build the index
idx = InvertedIndex()
for i, tokens in enumerate(tokenized):
    idx.add_document(i, tokens)

print(f"Index size: {len(idx.index)} unique terms\n")
print("Posting list for 'retrieval':")
print(f"  {idx.index['retrieval']}")
print(f"\nAND query 'retrieval search':  {idx.search_and('retrieval search')}")
print(f"OR  query 'retrieval search':  {idx.search_or('retrieval search')}")

In [ ]:
# Visualize posting list intersection
terms_to_show = ["retrieval", "search", "engines"]
fig, ax = plt.subplots(figsize=(8, 2.5))

for row, term in enumerate(terms_to_show):
    doc_ids = [doc_id for doc_id, _ in idx.index.get(term, [])]
    ax.scatter(doc_ids, [row] * len(doc_ids), s=100, zorder=3)
    ax.annotate(term, (-0.8, row), fontsize=10, va="center", fontweight="bold")

# Highlight intersection
common = set.intersection(*[
    {d for d, _ in idx.index.get(t, [])} for t in terms_to_show
])
for d in common:
    ax.axvline(d, color="red", alpha=0.3, linewidth=8)

ax.set(xlabel="Document ID", yticks=range(len(terms_to_show)),
       yticklabels=[""] * len(terms_to_show),
       title="Posting Lists & Intersection (red = docs matching all terms)")
ax.set_xlim(-1.5, len(corpus))
plt.tight_layout()
plt.show()

---
## 7 · Lexical vs Semantic Retrieval

BM25 and TF-IDF are **lexical** methods: they match exact tokens. This is
fast and precise when the user's query uses the same vocabulary as the
documents. But what happens when the vocabulary differs?

| Scenario | Lexical | Semantic |
|---|---|---|
| Query: "cat" → Doc: "cat sat on the mat" | ✅ Exact match | ✅ |
| Query: "feline" → Doc: "cat sat on the mat" | ❌ Vocabulary mismatch | ✅ Embedding similarity |
| Query: "AB-1234" → Doc: "AB-1234 specification" | ✅ Exact match | ❌ Rare token OOV |
| Query: long technical phrase | ✅ Precise | ⚠️ May dilute signal |

**Neither method dominates in all cases.** This motivates **hybrid retrieval**:
combine BM25 and semantic search, then fuse the rankings.

Let's run the same queries through both methods and compare.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use TF-IDF vectors as a lightweight proxy for embeddings
# (In production you would use a transformer model like in Notebook 02)
sem_vectorizer = TfidfVectorizer()
sem_matrix = sem_vectorizer.fit_transform(corpus)

def semantic_rank(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    q_vec = sem_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, sem_matrix).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return [(i, sims[i]) for i in top_idx]

test_queries = [
    "information retrieval search",   # exact vocabulary match
    "finding relevant documents",     # paraphrase / vocab mismatch
    "cat and dog",                    # concrete entities
]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: '{q}'")
    print(f"{'='*60}")
    print("\n  BM25 top-3:")
    for i, s in bm25.rank(q, top_k=3):
        print(f"    [{i}] {s:.3f}  {corpus[i]}")
    print("\n  Semantic (TF-IDF cosine) top-3:")
    for i, s in semantic_rank(q, top_k=3):
        print(f"    [{i}] {s:.3f}  {corpus[i]}")

### Observations

- **Exact vocabulary queries** ("information retrieval search"): both methods
  agree because the query terms appear literally in the relevant documents.
- **Paraphrase queries** ("finding relevant documents"): semantic retrieval
  can bridge the vocabulary gap, while BM25 struggles without shared terms.
  This is the classic **vocabulary mismatch** problem that plagued IR for decades.
- **Entity queries** ("cat and dog"): BM25 excels because entities are
  specific tokens best matched exactly. Embedding models sometimes dilute
  entity signal across a broader semantic neighbourhood.

This complementarity is why state-of-the-art RAG pipelines typically use
**hybrid retrieval**, combining a BM25 score with a dense embedding score
via techniques like Reciprocal Rank Fusion (RRF) or learned score combination.

---
## 8 · Evaluation Preview: Measuring Retrieval Quality

How do we know if our retrieval is good? Without quantitative evaluation,
we are flying blind. The IR community has developed a standard set of
metrics over decades of shared evaluation campaigns (TREC, CLEF, NTCIR).
The most commonly used metrics are:

- **Precision@k**: Of the top-$k$ results, what fraction is relevant?
  This measures the *quality* of the results the user actually sees.
- **Recall@k**: Of all relevant documents, what fraction appears in the top-$k$?
  This measures *completeness*—are we missing important documents?
- **MRR (Mean Reciprocal Rank)**: For each query, compute $1 / \text{rank of first relevant result}$, then average across queries.
  This focuses on how quickly the user finds *something* useful.

Let's define ground-truth relevance for our test queries and compute these
metrics for both BM25 and semantic retrieval.

In [ ]:
# Ground-truth relevant document indices for each test query
ground_truth = {
    "information retrieval search": {3, 4, 6},
    "finding relevant documents":   {3, 4, 6},
    "cat and dog":                  {0, 1, 7},
}

def precision_at_k(retrieved: list[int], relevant: set[int], k: int) -> float:
    top_k = retrieved[:k]
    return len(set(top_k) & relevant) / k

def recall_at_k(retrieved: list[int], relevant: set[int], k: int) -> float:
    top_k = retrieved[:k]
    return len(set(top_k) & relevant) / len(relevant) if relevant else 0.0

def reciprocal_rank(retrieved: list[int], relevant: set[int]) -> float:
    for rank, doc_id in enumerate(retrieved, 1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

k = 3
print(f"{'Query':<35} {'Method':<10} {'P@{}'.format(k):<8} {'R@{}'.format(k):<8} {'RR':<8}")
print("-" * 70)

mrr_bm25, mrr_sem = [], []

for q, rel in ground_truth.items():
    bm25_ids = [i for i, _ in bm25.rank(q, top_k=k)]
    sem_ids  = [i for i, _ in semantic_rank(q, top_k=k)]

    rr_b = reciprocal_rank(bm25_ids, rel)
    rr_s = reciprocal_rank(sem_ids, rel)
    mrr_bm25.append(rr_b)
    mrr_sem.append(rr_s)

    short_q = q[:33]
    print(f"{short_q:<35} {'BM25':<10} {precision_at_k(bm25_ids, rel, k):<8.2f} "
          f"{recall_at_k(bm25_ids, rel, k):<8.2f} {rr_b:<8.2f}")
    print(f"{'':35} {'Semantic':<10} {precision_at_k(sem_ids, rel, k):<8.2f} "
          f"{recall_at_k(sem_ids, rel, k):<8.2f} {rr_s:<8.2f}")

print(f"\nMRR — BM25: {np.mean(mrr_bm25):.3f}  |  Semantic: {np.mean(mrr_sem):.3f}")

---
## 🏋️ Exercises

### Exercise 1 — BM25 Parameter Tuning

Sweep $k_1 \in [0.5, 3.0]$ and $b \in [0.0, 1.0]$ and plot a heatmap of
MRR across the three test queries. Which combination works best for this corpus?

In [ ]:
# Exercise 1: BM25 parameter sweep
# YOUR CODE HERE
# k1_range = np.linspace(0.5, 3.0, 10)
# b_range = np.linspace(0.0, 1.0, 10)
# results = np.zeros((len(k1_range), len(b_range)))
# for i, k1 in enumerate(k1_range):
#     for j, b in enumerate(b_range):
#         model = BM25(tokenized, k1=k1, b=b)
#         ...
# plt.imshow(results, ...)

### Exercise 2 — Build a Hybrid Retriever

Combine BM25 and semantic scores using **Reciprocal Rank Fusion (RRF)**:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k = 60$ is a standard constant. Implement this and show that the
hybrid outperforms either individual method on the test queries.

In [ ]:
# Exercise 2: Hybrid retriever with RRF
# YOUR CODE HERE
# def rrf_fuse(rankings: list[list[int]], k: int = 60) -> list[tuple[int, float]]:
#     scores = defaultdict(float)
#     for ranking in rankings:
#         for rank, doc_id in enumerate(ranking, 1):
#             scores[doc_id] += 1.0 / (k + rank)
#     return sorted(scores.items(), key=lambda x: -x[1])

### Exercise 3 — Vocabulary Mismatch Experiment

Add five paraphrase queries (e.g., "feline" for "cat", "canine" for "dog")
and measure how badly BM25 degrades compared to a semantic model. Quantify
the gap in MRR.

In [ ]:
# Exercise 3: Vocabulary mismatch
# YOUR CODE HERE
# paraphrase_queries = {
#     "feline sitting on a rug":        {0},
#     "canine chasing animals":          {1, 2},
#     ...
# }

---
## 🔑 Key Takeaways

1. **TF-IDF** multiplies local term frequency by global rarity (IDF),
   producing a simple but effective relevance score.
2. **BM25** improves on TF-IDF with TF saturation ($k_1$) and document
   length normalization ($b$). It remains the default in Elasticsearch,
   Solr, and most production search systems.
3. **Inverted indices** flip the document→term mapping, enabling sub-linear
   query processing even on billion-document collections.
4. **Lexical and semantic retrieval are complementary**: BM25 excels at
   exact-match and entity queries; embeddings handle paraphrases and
   vocabulary gaps. Hybrid retrieval combines the best of both.
5. **Retrieval quality is the ceiling for RAG quality**—no generative model
   can compensate for missing context.

---
## What's Next

Now that you understand both embedding-based retrieval (Notebook 02) and
classical retrieval (this notebook), you have everything you need to build
a complete RAG pipeline. Head to:

→ **[rag/simple_rag.ipynb](../rag/simple_rag.ipynb)** — put retrieval +
  generation together for the first time.

---
## 📚 References

1. **Robertson, S. E., & Zaragoza, H.** (2009). *The Probabilistic Relevance
   Framework: BM25 and Beyond*. Foundations and Trends in Information
   Retrieval, 3(4), 333–389.
2. **Manning, C. D., Raghavan, P., & Schütze, H.** (2008). *Introduction to
   Information Retrieval*. Cambridge University Press.
   [Online edition](https://nlp.stanford.edu/IR-book/)
3. **Cormack, G. V., Clarke, C. L. A., & Buettcher, S.** (2009).
   *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank
   Learning Methods*. SIGIR 2009.